In [1]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24118743/versions/2 -O ../data/Chesseract.zip
!mkdir ../data/Chesseract
!unzip ../data/Chesseract.zip -d ../data/Chesseract/
!rm ../data/Chesseract.zip

--2025-05-19 19:13:51--  https://data.ncl.ac.uk/ndownloader/articles/24118743/versions/2
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 54.154.70.109, 3.254.11.99, 108.128.85.87
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|54.154.70.109|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215698081 (206M) [application/zip]
Saving to: ‘../data/Chesseract.zip’

../data/Chesseract. 100%[===================>] 205.71M  3.22MB/s    in 39s     

2025-05-19 19:14:31 (5.30 MB/s) - ‘../data/Chesseract.zip’ saved [215698081/215698081]

mkdir: cannot create directory ‘../data/Chesseract’: File exists
Archive:  ../data/Chesseract.zip
replace ../data/Chesseract/test_x.npy? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from senmodel.model.utils import *
from senmodel.metrics.nonlinearity_metrics import *
from senmodel.metrics.edge_finder import *
from senmodel.metrics.train_metrics import *
from senmodel.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=12 * 8 * 8, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [3]:
hyperparams = {
    "num_epochs": 26,
    "batch_size": 256,
    "metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "aggregation_mode": "mean",
    "choose_thresholds": {"fc0": 0.05}, # 1.0 -> no edges, 0.0 -> all edges
    "choose_thresholds_del": {"fc0": 0.99}, # 1.0 -> no edges, 0.0 -> all edges
    "threshold": 0.5,
    "min_delta_epoch_replace": 24,
    "window_size": 5,
    "lr": 2e-4,
    "delete_after": 4,    
    "task_type": "classification",
    "fully_connected": False,
    "max_to_replace": 3000, # None -> no limit
    "weight_decay": 1e-3,
}

name = ", ".join(
    f"{key}: {value.__class__.__name__ if key == 'metric' else value}"
    for key, value in hyperparams.items()
)

name

"num_epochs: 26, batch_size: 256, metric: AbsGradientEdgeMetric, aggregation_mode: mean, choose_thresholds: {'fc0': 0.05}, choose_thresholds_del: {'fc0': 0.99}, threshold: 0.5, min_delta_epoch_replace: 24, window_size: 5, lr: 0.0002, delete_after: 4, task_type: classification, fully_connected: False, max_to_replace: 3000, weight_decay: 0.001"

In [4]:
train_x = np.load("../data/Chesseract/train_x.npy")
train_y = np.load("../data/Chesseract/train_y.npy")
valid_x = np.load("../data/Chesseract/valid_x.npy")
valid_y = np.load("../data/Chesseract/valid_y.npy")
test_x = np.load('../data/Chesseract/test_x.npy')
test_y = np.load('../data/Chesseract/test_y.npy')


train_tensor_x = torch.tensor(train_x).float().view(-1, 12 * 8 * 8)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 12 * 8 * 8)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 12 * 8 * 8)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, worker_init_fn=np.random.seed(0),num_workers=0, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, worker_init_fn=np.random.seed(0),num_workers=0, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, worker_init_fn=np.random.seed(0),num_workers=0, generator=g)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
import wandb

wandb.login()

wandb: Currently logged in as: vanyamironov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-chesseract",
    name=f"local-cudnn",
    config=hyperparams
)

In [8]:
import time
start_time = time.time()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(sparse_model.parameters(), lr=hyperparams['lr'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:02<00:00, 81.52it/s]


Epoch 1/26, Train Loss: 1.9832, Val Loss: 1.7389, Val Accuracy: 0.3618


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 100.02it/s]


Epoch 2/26, Train Loss: 1.5979, Val Loss: 1.4833, Val Accuracy: 0.4136


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 136.02it/s]


Epoch 3/26, Train Loss: 1.4102, Val Loss: 1.3519, Val Accuracy: 0.4482


... (42 outputs left out)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 118.23it/s]


Epoch 25/26, Train Loss: 0.9624, Val Loss: 1.0102, Val Accuracy: 0.5201


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:01<00:00, 183.72it/s]


Epoch 26/26, Train Loss: 0.9590, Val Loss: 1.0077, Val Accuracy: 0.5217
Chosen edges: tensor([[  0,   0,   0,  ...,   2,   2,   2],
        [  8,   9,  10,  ..., 756, 757, 758]], device='cuda:0') 1967
--- 417.72414445877075 seconds ---


In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [10]:
accuracy, params

(0.5849584958495849, 1338593)

In [ ]:
torch.save(sparse_model, 'chesseract.pt')

In [11]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 57.83, 57.45, 55.69, 52.74, 56.26, 59.60, 55.15, 57.20, 58.24, 60.76, 59.16, 68.83, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))
styled_table

Model,Accuracy (%),Parameters
Ours,58.50,"1,338,593.0"
ResNet-18,57.83,"11,689,512.0"
AlexNet,57.45,"61,100,840.0"
VGG16,55.69,"138,357,544.0"
ConvNext,52.74,"88,591,464.0"
MNASNet,56.26,"4,383,312.0"
DenseNet,59.60,"28,681,000.0"
ResNeXt,55.15,"25,028,904.0"
PC-DARTS,57.20,nan
DrNAS,58.24,nan
